In [ ]:
import spacy
nlp = spacy.load('en_core_web_md')

In [ ]:
with open('../data/hamlet.txt') as p:
    # replace em-dashes so spacy can tokenize properly
    hamlet = p.read().replace("—", " — ")

doc = nlp(hamlet)
print(doc)

In [ ]:
from collections import deque
from itertools import islice

def sliding_window(iterable, n):
    "Collect data into overlapping fixed-length chunks or blocks."
    # sliding_window('ABCDEFG', 3) → ABC BCD CDE DEF EFG
    iterator = iter(iterable)
    window = deque(islice(iterator, n - 1), maxlen=n)
    for x in iterator:
        window.append(x)
        yield tuple(window)


In [ ]:
shopping_cart = {'VERB', 'NOUN', 'ADJ', 'ADV'} # POS(part of sentence) filter

def is_good(token):
    text = token.text # to make it all a string so no errors occur with string only functions
    if text[0] == "’": # no ’s or ’d or something
        return False
    return any(char.isalpha() for char in text)
    # checks if there are any real letters to remove garbage (still keeps o'er)

In [ ]:
# gets good words using the filter
goods = [token.text.lower() for token in doc if token.pos_ in shopping_cart
                    and is_good(token)]
goodies = list(goods)
print(goodies)

In [ ]:
from typing import Any

windows: list[tuple[Any, ...]] = list(sliding_window(goodies, 11)) # makes windows
print(windows)

In [ ]:
from collections import defaultdict, Counter
from itertools import combinations

cooccurrence = defaultdict(Counter) # creates match dictionary of center word and context words
# change it to set middle word as center words.len()/2

for words in windows:
    center = int(len(words)/2)
    # words = {token.text.lower() for token in window} # gets all words per window
    center_word = words[center]
    for index, context in enumerate(words):
        if index == center:
            continue
        cooccurrence[center_word][context] += 1
print(cooccurrence)


In [ ]:
import math

total_pairs = sum(sum(n.values()) for n in cooccurrence.values())
print(total_pairs)

In [ ]:
count = Counter(goodies) # gets total count of each word
total_words = sum(count.values())
print(total_words)

In [ ]:
def pmi(w, c) -> float | None:
    w, c = w.lower(), c.lower()
    if cooccurrence[w][c] == 0:
        return None
    p_wc = cooccurrence[w][c] / total_pairs # probability of a given pair
    p_w = count[w] / total_words # probability of center word
    p_c = count[c] / total_words # probability of context word
    result = math.log2(p_wc / (p_w * math.pow(p_c, .75))) # p(pair) / P(word U context) if independent
    return None if result < 0 else float(result)

pmi('strength', 'life')


In [ ]:
import random
random_word_sample = random.sample(goodies, k=10)
print(random_word_sample)

In [ ]:
def get_close_words(word) -> list[tuple[str, float]]:
    closest_words = []
    for tok in cooccurrence[word]:
        if count[word] < 5:
            continue
        closeness = pmi(word, tok)
        if closeness:
            closest_words.append((closeness, tok))
    closest_words.sort(reverse=True)
    return [(tok, closeness) for closeness, tok in closest_words[:10]]


In [ ]:
get_close_words("life")

